# Basic LSTM with attention

In [20]:
# ============================
# TRAINING + SAVE MODEL
# ============================

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr, spearmanr
from torch.utils.data import Dataset, DataLoader
import re

# ============================
# CONFIG
# ============================

MAX_LEN = 60
EMBED_DIM = 100
HIDDEN_DIM = 128
BATCH_SIZE = 32
EPOCHS = 20
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================
# CLEANING
# ============================

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text

def tokenize(text):
    return text.split()

# ============================
# LOAD DATA
# ============================

df = pd.read_csv("/kaggle/input/datasets/uppulurimadhuri/semeval2013-task-7-sag/Training_data.csv")

df = df.rename(columns={
    "Question_Text": "question",
    "Reference_Answer_Text": "passage",
    "Student_Answer_Text": "student",
    "Student_Answer_Accuracy": "label"
})

df["question"] = df["question"].apply(clean_text)
df["passage"] = df["passage"].apply(clean_text)
df["student"] = df["student"].apply(clean_text)

# ============================
# LABEL MAP
# ============================

df["label"] = df["label"].astype(str).str.lower()

label_map = {
    "correct": 1.0,
    "partially_correct": 0.7,
    "partially_correct_incomplete": 0.6,
    "contradictory": 0.1,
    "irrelevant": 0.0,
    "non_domain": 0.0
}

df["score"] = df["label"].map(label_map)
df = df.dropna(subset=["score"])

# ============================
# BUILD VOCAB
# ============================

vocab = {"<pad>": 0, "<unk>": 1}

def build_vocab(texts):
    idx = 2
    for text in texts:
        for word in tokenize(text):
            if word not in vocab:
                vocab[word] = idx
                idx += 1

build_vocab(df["question"])
build_vocab(df["passage"])
build_vocab(df["student"])

def encode(text):
    tokens = tokenize(text)
    ids = [vocab.get(w, 1) for w in tokens]
    if len(ids) < MAX_LEN:
        ids += [0]*(MAX_LEN-len(ids))
    else:
        ids = ids[:MAX_LEN]
    return ids

df["q_ids"] = df["question"].apply(encode)
df["p_ids"] = df["passage"].apply(encode)
df["s_ids"] = df["student"].apply(encode)

# ============================
# DATASET
# ============================

class JointDataset(Dataset):
    def __init__(self, df):
        self.q = df["q_ids"].tolist()
        self.p = df["p_ids"].tolist()
        self.s = df["s_ids"].tolist()
        self.score = df["score"].tolist()

    def __len__(self):
        return len(self.score)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.q[idx]),
            torch.tensor(self.p[idx]),
            torch.tensor(self.s[idx]),
            torch.tensor(self.score[idx], dtype=torch.float)
        )

train_df, test_df = train_test_split(df, test_size=0.2)

train_loader = DataLoader(JointDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(JointDataset(test_df), batch_size=BATCH_SIZE)

# ============================
# MODEL
# ============================

class JointModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=0)
        self.lstm = nn.LSTM(EMBED_DIM, HIDDEN_DIM, batch_first=True, bidirectional=True)
        self.attention = nn.Linear(HIDDEN_DIM*2, HIDDEN_DIM*2)

    def encode(self, x):
        x = self.embedding(x)
        outputs, _ = self.lstm(x)
        return outputs

    def attention_layer(self, passage_enc, question_vec):
        scores = torch.bmm(passage_enc, question_vec.unsqueeze(2)).squeeze(2)
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(passage_enc * weights.unsqueeze(2), dim=1)
        return context

    def forward(self, q, p, s):
        q_enc = self.encode(q)
        p_enc = self.encode(p)
        s_enc = self.encode(s)

        q_vec = torch.mean(q_enc, dim=1)
        evidence = self.attention_layer(p_enc, q_vec)
        s_vec = torch.mean(s_enc, dim=1)

        sim = nn.functional.cosine_similarity(evidence, s_vec)
        return sim

model = JointModel(len(vocab)).to(DEVICE)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ============================
# TRAIN
# ============================

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for q, p, s, score in train_loader:
        q, p, s, score = q.to(DEVICE), p.to(DEVICE), s.to(DEVICE), score.to(DEVICE)

        optimizer.zero_grad()
        preds = model(q, p, s)
        loss = criterion(preds, score)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

# ============================
# SAVE MODEL
# ============================

torch.save({
    "model_state": model.state_dict(),
    "vocab": vocab
}, "asag_model_v1.pth")

print("Model saved")

Epoch 1, Loss: 18.9325
Epoch 2, Loss: 13.6226
Epoch 3, Loss: 11.0845
Epoch 4, Loss: 9.1082
Epoch 5, Loss: 7.7794
Epoch 6, Loss: 6.5509
Epoch 7, Loss: 5.6022
Epoch 8, Loss: 4.7472
Epoch 9, Loss: 4.0688
Epoch 10, Loss: 3.4011
Epoch 11, Loss: 3.0165
Epoch 12, Loss: 2.6315
Epoch 13, Loss: 2.3463
Epoch 14, Loss: 2.0085
Epoch 15, Loss: 1.6548
Epoch 16, Loss: 1.4120
Epoch 17, Loss: 1.2454
Epoch 18, Loss: 1.1448
Epoch 19, Loss: 1.0413
Epoch 20, Loss: 0.9847
Model saved


In [21]:
# ============================
# LOAD MODEL + TEST
# ============================

import torch
import torch.nn as nn
import re

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================
# LOAD
# ============================

checkpoint = torch.load("/kaggle/working/asag_model_v1.pth", map_location=DEVICE)
vocab = checkpoint["vocab"]

MAX_LEN = 60
EMBED_DIM = 100
HIDDEN_DIM = 128

# ============================
# MODEL
# ============================

class JointModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=0)
        self.lstm = nn.LSTM(EMBED_DIM, HIDDEN_DIM, batch_first=True, bidirectional=True)
        self.attention = nn.Linear(HIDDEN_DIM*2, HIDDEN_DIM*2)

    def encode(self, x):
        x = self.embedding(x)
        outputs, _ = self.lstm(x)
        return outputs

    def attention_layer(self, passage_enc, question_vec):
        scores = torch.bmm(passage_enc, question_vec.unsqueeze(2)).squeeze(2)
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(passage_enc * weights.unsqueeze(2), dim=1)
        return context

    def forward(self, q, p, s):
        q_enc = self.encode(q)
        p_enc = self.encode(p)
        s_enc = self.encode(s)

        q_vec = torch.mean(q_enc, dim=1)
        evidence = self.attention_layer(p_enc, q_vec)
        s_vec = torch.mean(s_enc, dim=1)

        sim = nn.functional.cosine_similarity(evidence, s_vec)
        return sim

model = JointModel(len(vocab)).to(DEVICE)
model.load_state_dict(checkpoint["model_state"])
model.eval()

print("Model loaded")

# ============================
# UTILS
# ============================

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text

def tokenize(text):
    return text.split()

def encode(text):
    tokens = tokenize(text)
    ids = [vocab.get(w, 1) for w in tokens]
    if len(ids) < MAX_LEN:
        ids += [0]*(MAX_LEN-len(ids))
    else:
        ids = ids[:MAX_LEN]
    return ids

# ============================
# PREDICT
# ============================

def predict(question, passage, student, max_marks=5):
    q = torch.tensor([encode(clean_text(question))]).to(DEVICE)
    p = torch.tensor([encode(clean_text(passage))]).to(DEVICE)
    s = torch.tensor([encode(clean_text(student))]).to(DEVICE)

    with torch.no_grad():
        sim = model(q, p, s).item()

    return {
        "similarity": sim,
        "marks": max(0, min(sim, 1)) * max_marks
    }

# ============================
# INTERACTIVE
# ============================

def interactive():
    print("\n=== Joint Model Grader ===\n")
    while True:
        q = input("Question: ")
        if q == "exit":
            break
        p = input("Passage: ")
        s = input("Student Answer: ")
        m = float(input("Max Marks: "))

        res = predict(q, p, s, m)

        print("\n--- Result ---")
        print("Similarity:", round(res["similarity"], 4))
        print("Marks:", round(res["marks"], 2), "/", m)
        print("----------------\n")

interactive()

Model loaded

=== Joint Model Grader ===



Question:  Where did they live?
Passage:  The two ducks lived in a peaceful pond filled with fish and plants. It was a safe and happy place for them.
Student Answer:  In a pond
Max Marks:  3



--- Result ---
Similarity: 0.7231
Marks: 2.17 / 3.0
----------------



Question:  Why can dependence on one customer be risky?
Passage:  A minority business that depends heavily on one large corporate client may struggle if that client stops giving orders, making the business unstable.
Student Answer:  business becomes unstable
Max Marks:  3



--- Result ---
Similarity: 0.5288
Marks: 1.59 / 3.0
----------------



Question:  What do plants need to make food?
Passage:  Plants require sunlight, water, and carbon dioxide to produce food through photosynthesis.
Student Answer:  food and water
Max Marks:  3



--- Result ---
Similarity: 0.2119
Marks: 0.64 / 3.0
----------------



Question:  How did corporate support change over time?
Passage:  Corporate contracts with minority businesses increased significantly from $77 million in 1972 to over $1.1 billion in 1977, showing growing support.
Student Answer:  Increased over time
Max Marks:  2



--- Result ---
Similarity: 0.2903
Marks: 0.58 / 2.0
----------------



Question:  exit


# LSTM Enhanced with glove

In [9]:
# ============================
# CELL 1: TRAIN + SAVE
# ============================

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import re, os

# CONFIG
MAX_LEN = 120
EMBED_DIM = 100
HIDDEN_DIM = 128
BATCH_SIZE = 32
EPOCHS = 30
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GLOVE_PATH = "/kaggle/input/datasets/rtatman/glove-global-vectors-for-word-representation/glove.6B.100d.txt"
SAVE_PATH = "asag_model.pth"

# CLEAN
def clean_text(text):
    return re.sub(r'[^a-z0-9\s]', '', str(text).lower())

def tokenize(text):
    return text.split()

# LOAD DATA
df = pd.read_csv("/kaggle/input/datasets/uppulurimadhuri/semeval2013-task-7-sag/Training_data.csv")

df = df.rename(columns={
    "Question_Text": "question",
    "Reference_Answer_Text": "passage",
    "Student_Answer_Text": "student",
    "Student_Answer_Accuracy": "label"
})

df["question"] = df["question"].apply(clean_text)
df["passage"] = df["passage"].apply(clean_text)
df["student"] = df["student"].apply(clean_text)

# LABEL MAP
label_map = {
    "correct": 1.0,
    "partially_correct": 0.7,
    "partially_correct_incomplete": 0.6,
    "contradictory": 0.1,
    "irrelevant": 0.0,
    "non_domain": 0.0
}

df["label"] = df["label"].astype(str).str.lower()
df["score"] = df["label"].map(label_map)
df = df.dropna(subset=["score"])

# VOCAB
vocab = {"<pad>": 0, "<unk>": 1}

def build_vocab(texts):
    idx = 2
    for text in texts:
        for w in tokenize(text):
            if w not in vocab:
                vocab[w] = idx
                idx += 1

build_vocab(df["question"])
build_vocab(df["passage"])
build_vocab(df["student"])

def encode(text):
    ids = [vocab.get(w, 1) for w in tokenize(text)]
    return (ids + [0]*MAX_LEN)[:MAX_LEN]

df["q_ids"] = df["question"].apply(encode)
df["p_ids"] = df["passage"].apply(encode)
df["s_ids"] = df["student"].apply(encode)

# LOAD GLOVE
def load_glove(vocab, path):
    emb = np.random.normal(scale=0.6, size=(len(vocab), EMBED_DIM))

    if not os.path.exists(path):
        return torch.tensor(emb, dtype=torch.float)

    glove = {}
    with open(path, encoding="utf8") as f:
        for line in f:
            parts = line.split()
            glove[parts[0]] = np.array(parts[1:], dtype=float)

    for w, i in vocab.items():
        if w in glove:
            emb[i] = glove[w]

    return torch.tensor(emb, dtype=torch.float)

embedding_matrix = load_glove(vocab, GLOVE_PATH)

# DATASET
class DatasetASAG(Dataset):
    def __init__(self, df):
        self.q = df["q_ids"].tolist()
        self.p = df["p_ids"].tolist()
        self.s = df["s_ids"].tolist()
        self.y = df["score"].tolist()

    def __len__(self): return len(self.y)

    def __getitem__(self, i):
        return (
            torch.tensor(self.q[i]),
            torch.tensor(self.p[i]),
            torch.tensor(self.s[i]),
            torch.tensor(self.y[i], dtype=torch.float)
        )

train_df, _ = train_test_split(df, test_size=0.2)
train_loader = DataLoader(DatasetASAG(train_df), batch_size=BATCH_SIZE, shuffle=True)

# MODEL
class Model(nn.Module):
    def __init__(self, vocab_size, emb_matrix):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=0)
        self.embedding.weight.data.copy_(emb_matrix)
        self.lstm = nn.LSTM(EMBED_DIM, HIDDEN_DIM, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(HIDDEN_DIM*2, 1)

    def encode(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        return out

    def attention(self, seq, q_vec):
        scores = torch.bmm(seq, q_vec.unsqueeze(2)).squeeze(2)
        w = torch.softmax(scores, dim=1)
        ctx = torch.sum(seq * w.unsqueeze(2), dim=1)
        return ctx

    def forward(self, q, p, s):
        q_vec = torch.mean(self.encode(q), dim=1)
        evidence = self.attention(self.encode(p), q_vec)
        student = self.attention(self.encode(s), q_vec)

        diff = torch.abs(evidence - student)
        return torch.sigmoid(self.fc(diff)).squeeze()

model = Model(len(vocab), embedding_matrix).to(DEVICE)

# TRAIN
optimizer = optim.Adam(model.parameters(), lr=0.001)

for ep in range(EPOCHS):
    model.train()
    total = 0

    for q,p,s,y in train_loader:
        q,p,s,y = q.to(DEVICE), p.to(DEVICE), s.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        pred = model(q,p,s)
        loss = ((pred - y)**2).mean()
        loss.backward()
        optimizer.step()

        total += loss.item()

    print(f"Epoch {ep+1}, Loss: {total:.4f}")

# SAVE
torch.save({
    "model_state": model.state_dict(),
    "vocab": vocab,
    "config": {
        "MAX_LEN": MAX_LEN,
        "EMBED_DIM": EMBED_DIM,
        "HIDDEN_DIM": HIDDEN_DIM
    }
}, SAVE_PATH)

print(" Model saved")

Epoch 1, Loss: 21.9542
Epoch 2, Loss: 20.7895
Epoch 3, Loss: 18.6836
Epoch 4, Loss: 16.5935
Epoch 5, Loss: 14.6159
Epoch 6, Loss: 12.7821
Epoch 7, Loss: 11.0750
Epoch 8, Loss: 9.2149
Epoch 9, Loss: 7.3625
Epoch 10, Loss: 5.9372
Epoch 11, Loss: 4.6355
Epoch 12, Loss: 3.7043
Epoch 13, Loss: 2.9538
Epoch 14, Loss: 2.2581
Epoch 15, Loss: 1.9816
Epoch 16, Loss: 1.7967
Epoch 17, Loss: 1.4722
Epoch 18, Loss: 1.2902
Epoch 19, Loss: 1.1783
Epoch 20, Loss: 1.0886
Epoch 21, Loss: 1.1796
Epoch 22, Loss: 1.0365
Epoch 23, Loss: 1.0305
Epoch 24, Loss: 1.0067
Epoch 25, Loss: 0.9311
Epoch 26, Loss: 0.9223
Epoch 27, Loss: 0.8567
Epoch 28, Loss: 0.9608
Epoch 29, Loss: 0.9581
Epoch 30, Loss: 0.9675
 Model saved


In [ ]:
# ============================
# CELL 2: LOAD + TEST
# ============================

import torch
import torch.nn as nn
import re

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# LOAD
ckpt = torch.load("asag_model.pth", map_location=DEVICE)

vocab = ckpt["vocab"]
config = ckpt["config"]

MAX_LEN = config["MAX_LEN"]
EMBED_DIM = config["EMBED_DIM"]
HIDDEN_DIM = config["HIDDEN_DIM"]

# MODEL
class Model(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=0)
        self.lstm = nn.LSTM(EMBED_DIM, HIDDEN_DIM, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(HIDDEN_DIM*2, 1)

    def encode(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        return out

    def attention(self, seq, q_vec):
        scores = torch.bmm(seq, q_vec.unsqueeze(2)).squeeze(2)
        w = torch.softmax(scores, dim=1)
        ctx = torch.sum(seq * w.unsqueeze(2), dim=1)
        return ctx

    def forward(self, q, p, s):
        q_vec = torch.mean(self.encode(q), dim=1)
        evidence = self.attention(self.encode(p), q_vec)
        student = self.attention(self.encode(s), q_vec)

        diff = torch.abs(evidence - student)
        return torch.sigmoid(self.fc(diff)).squeeze()

model = Model(len(vocab)).to(DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()

print(" Model loaded")

# UTILS
def clean_text(text):
    return re.sub(r'[^a-z0-9\s]', '', str(text).lower())

def tokenize(text):
    return text.split()

def encode(text):
    ids = [vocab.get(w, 1) for w in tokenize(text)]
    return (ids + [0]*MAX_LEN)[:MAX_LEN]

def extract_best_sentence(q, p):
    sentences = p.split('.')
    q_words = set(tokenize(q))

    best = ""
    best_score = 0

    for s in sentences:
        score = len(set(tokenize(s)) & q_words)
        if score > best_score:
            best_score = score
            best = s

    return best.strip()

# PREDICT
def predict(q,p,s,max_marks=5):
    q,p,s = clean_text(q), clean_text(p), clean_text(s)
    best = extract_best_sentence(q,p)

    q_t = torch.tensor([encode(q)]).to(DEVICE)
    p_t = torch.tensor([encode(best)]).to(DEVICE)
    s_t = torch.tensor([encode(s)]).to(DEVICE)

    with torch.no_grad():
        sim = model(q_t,p_t,s_t).item()

    overlap = len(set(tokenize(best)) & set(tokenize(s))) / (len(set(tokenize(s)))+1e-5)
    score = (0.5*sim + 0.5*overlap)*max_marks

    return round(score,2), best

# INTERACTIVE
while True:
    q = input("Question: ")
    if q == "exit":
        break
    p = input("Passage: ")
    s = input("Student Answer: ")
    m = float(input("Max Marks: "))

    score, sent = predict(q,p,s,m)

    print("\nMarks:", score, "/", m)
    print("Relevant Sentence:", sent)
    print("----------------\n")

# LSTM WITH CSV INPUT

In [16]:
# ============================================
# CELL 1: TRAINING AND SAVING THE MODEL
# Using the actual SAG dataset from Training_data.csv
# ============================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr, spearmanr
import re
import string
from collections import Counter
import pickle
import os

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# ============================================
# CONFIGURATION
# ============================================
MAX_SEQUENCE_LENGTH = 150
MAX_VOCAB_SIZE = 30000
EMBEDDING_DIM = 100  # Using 100-dim GloVe if available
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.3
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 0.001

# ============================================
# DATA PREPROCESSING PIPELINE
# ============================================

class TextPreprocessor:
    def __init__(self, max_vocab_size=MAX_VOCAB_SIZE, max_sequence_length=MAX_SEQUENCE_LENGTH):
        self.max_vocab_size = max_vocab_size
        self.max_sequence_length = max_sequence_length
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
        self.vocab_size = 2
        
    def clean_text(self, text):
        """Lowercase and remove punctuation"""
        if pd.isna(text):
            text = ""
        text = str(text).lower()
        text = re.sub(f'[{re.escape(string.punctuation)}]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    
    def build_vocabulary(self, texts):
        """Build vocabulary from list of texts"""
        word_counter = Counter()
        for text in texts:
            words = self.clean_text(text).split()
            word_counter.update(words)
        
        # Add most common words to vocabulary
        most_common = word_counter.most_common(self.max_vocab_size - 2)
        for word, _ in most_common:
            if word not in self.word2idx:
                self.word2idx[word] = self.vocab_size
                self.idx2word[self.vocab_size] = word
                self.vocab_size += 1
                
    def text_to_sequence(self, text):
        """Convert text to sequence of indices"""
        words = self.clean_text(text).split()
        sequence = [self.word2idx.get(word, self.word2idx['<UNK>']) for word in words]
        # Pad or truncate
        if len(sequence) > self.max_sequence_length:
            sequence = sequence[:self.max_sequence_length]
        else:
            sequence = sequence + [self.word2idx['<PAD>']] * (self.max_sequence_length - len(sequence))
        return sequence


# ============================================
# LOAD AND PREPROCESS SAG DATASET
# ============================================

def load_sag_dataset(csv_path='Training_data.csv'):
    """Load the SAG dataset from CSV file"""
    df = pd.read_csv(csv_path)
    
    # Map the accuracy labels to scores
    # Based on the data, we have categories like: correct, partially_correct_incomplete, contradictory, irrelevant, non_domain
    label_to_score = {
        'correct': 1.0,
        'partially_correct_incomplete': 0.7,
        'contradictory': 0.3,
        'irrelevant': 0.0,
        'non_domain': 0.0
    }
    
    # Create a clean dataframe
    data = []
    for idx, row in df.iterrows():
        # Skip rows with missing essential data
        if pd.isna(row['Question_Text']) or pd.isna(row['Reference_Answer_Text']) or pd.isna(row['Student_Answer_Text']):
            continue
            
        score = label_to_score.get(row['Student_Answer_Accuracy'], 0.0)
        
        data.append({
            'question_id': row['Question_ID'],
            'question': row['Question_Text'],
            'reference_answer': row['Reference_Answer_Text'],
            'student_answer': row['Student_Answer_Text'],
            'label': row['Student_Answer_Accuracy'],
            'score': score
        })
    
    df_clean = pd.DataFrame(data)
    print(f"Loaded {len(df_clean)} valid samples")
    print(f"Score distribution:\n{df_clean['score'].value_counts().sort_index()}")
    
    return df_clean


# ============================================
# PYTORCH DATASET
# ============================================

class SAGDataset(Dataset):
    def __init__(self, questions, reference_answers, student_answers, scores, preprocessor):
        self.preprocessor = preprocessor
        # Combine question and reference answer for encoder A
        self.encoder_a_inputs = [f"{q} {ref}" for q, ref in zip(questions, reference_answers)]
        self.encoder_b_inputs = student_answers
        self.scores = scores
        
    def __len__(self):
        return len(self.scores)
    
    def __getitem__(self, idx):
        seq_a = self.preprocessor.text_to_sequence(self.encoder_a_inputs[idx])
        seq_b = self.preprocessor.text_to_sequence(self.encoder_b_inputs[idx])
        score = self.scores[idx]
        
        return torch.tensor(seq_a, dtype=torch.long), \
               torch.tensor(seq_b, dtype=torch.long), \
               torch.tensor(score, dtype=torch.float)


# ============================================
# SIAMESE LSTM MODEL
# ============================================

class SiameseLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=EMBEDDING_DIM, hidden_dim=HIDDEN_DIM, 
                 num_layers=NUM_LAYERS, dropout=DROPOUT, pretrained_embeddings=None):
        super(SiameseLSTM, self).__init__()
        
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        
        # Embedding layer
        if pretrained_embeddings is not None:
            self.embedding = nn.Embedding.from_pretrained(pretrained_embeddings, freeze=False)
        else:
            self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
            nn.init.uniform_(self.embedding.weight, -0.1, 0.1)
            self.embedding.weight.data[0] = torch.zeros(embedding_dim)  # Pad token
        
        # Shared LSTM encoder
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Fully connected layer to reduce dimension for similarity
        self.fc = nn.Linear(hidden_dim * 2, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        
    def encode(self, x):
        """Encode input sequence to embedding"""
        embedded = self.dropout(self.embedding(x))
        lstm_out, (hidden, cell) = self.lstm(embedded)
        
        # Use concatenation of forward and backward last hidden states
        hidden_forward = hidden[-2, :, :]  # Last layer forward
        hidden_backward = hidden[-1, :, :]  # Last layer backward
        hidden_combined = torch.cat((hidden_forward, hidden_backward), dim=1)
        
        # Reduce dimension
        embedding = self.fc(hidden_combined)
        embedding = torch.tanh(embedding)  # Normalize
        return embedding
    
    def forward(self, input_a, input_b):
        """Forward pass through both encoders and compute similarity"""
        embedding_a = self.encode(input_a)
        embedding_b = self.encode(input_b)
        
        # Cosine similarity
        cos_sim = nn.functional.cosine_similarity(embedding_a, embedding_b, dim=1)
        # Ensure output is in [0, 1]
        cos_sim = (cos_sim + 1) / 2
        
        return cos_sim


# ============================================
# LOAD GLOVE EMBEDDINGS (Optional)
# ============================================

def load_glove_embeddings(glove_path, word2idx, embedding_dim=100):
    """Load GloVe embeddings"""
    embeddings = np.random.randn(len(word2idx), embedding_dim) * 0.01
    embeddings[0] = 0  # Padding token
    
    if os.path.exists(glove_path):
        print(f"Loading GloVe embeddings from {glove_path}...")
        with open(glove_path, 'r', encoding='utf-8') as f:
            for line in f:
                values = line.split()
                word = values[0]
                if word in word2idx:
                    idx = word2idx[word]
                    vec = np.array(values[1:], dtype='float32')
                    if len(vec) == embedding_dim:
                        embeddings[idx] = vec
        print("GloVe embeddings loaded successfully")
    else:
        print(f"GloVe file not found at {glove_path}, using random embeddings")
    
    return torch.tensor(embeddings, dtype=torch.float)


# ============================================
# TRAINING FUNCTION
# ============================================

def train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LEARNING_RATE, device='cuda'):
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
    
    best_val_loss = float('inf')
    best_val_pearson = 0
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0
        for batch_idx, (input_a, input_b, scores) in enumerate(train_loader):
            input_a, input_b, scores = input_a.to(device), input_b.to(device), scores.to(device)
            
            optimizer.zero_grad()
            outputs = model(input_a, input_b)
            loss = criterion(outputs, scores)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)
        
        # Validation phase
        model.eval()
        val_loss = 0
        all_preds = []
        all_scores = []
        
        with torch.no_grad():
            for input_a, input_b, scores in val_loader:
                input_a, input_b, scores = input_a.to(device), input_b.to(device), scores.to(device)
                outputs = model(input_a, input_b)
                val_loss += criterion(outputs, scores).item()
                all_preds.extend(outputs.cpu().numpy())
                all_scores.extend(scores.cpu().numpy())
        
        avg_val_loss = val_loss / len(val_loader)
        
        # Calculate metrics
        pearson_corr, _ = pearsonr(all_scores, all_preds)
        spearman_corr, _ = spearmanr(all_scores, all_preds)
        mse = mean_squared_error(all_scores, all_preds)
        
        print(f'Epoch {epoch+1}/{epochs}')
        print(f'  Train Loss: {avg_train_loss:.4f}')
        print(f'  Val Loss: {avg_val_loss:.4f}')
        print(f'  Pearson: {pearson_corr:.4f}, Spearman: {spearman_corr:.4f}, MSE: {mse:.4f}')
        
        # Save best model based on validation loss
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_val_pearson = pearson_corr
            torch.save({
                'model_state_dict': model.state_dict(),
                'vocab_size': model.embedding.num_embeddings,
                'embedding_dim': EMBEDDING_DIM,
                'hidden_dim': HIDDEN_DIM,
                'num_layers': NUM_LAYERS,
                'best_val_loss': best_val_loss,
                'best_val_pearson': best_val_pearson
            }, 'best_siamese_lstm_model.pth')
            print(f'  Saved best model with val loss: {best_val_loss:.4f}, Pearson: {best_val_pearson:.4f}')
        
        scheduler.step(avg_val_loss)
        print()
    
    return model


# ============================================
# MAIN TRAINING PIPELINE
# ============================================

def main_training(csv_path='Training_data.csv', glove_path='/kaggle/input/glove6b100d/glove.6B.100d.txt'):
    print("=" * 60)
    print("LOADING DATASET")
    print("=" * 60)
    df = load_sag_dataset(csv_path)
    
    # Prepare text data for vocabulary building
    all_texts = []
    for q, ref, student in zip(df['question'], df['reference_answer'], df['student_answer']):
        all_texts.append(f"{q} {ref}")
        all_texts.append(student)
    
    # Initialize preprocessor
    preprocessor = TextPreprocessor(max_sequence_length=MAX_SEQUENCE_LENGTH)
    preprocessor.build_vocabulary(all_texts)
    
    print(f"\nVocabulary size: {preprocessor.vocab_size}")
    print(f"Max sequence length: {MAX_SEQUENCE_LENGTH}")
    
    # Split data
    X_combined = list(zip(df['question'], df['reference_answer'], df['student_answer']))
    y = df['score'].values
    
    train_idx, val_idx = train_test_split(range(len(df)), test_size=0.2, random_state=42, stratify=df['score'].apply(lambda x: round(x*10)))
    
    train_questions = df['question'].iloc[train_idx].tolist()
    train_references = df['reference_answer'].iloc[train_idx].tolist()
    train_students = df['student_answer'].iloc[train_idx].tolist()
    train_scores = df['score'].iloc[train_idx].tolist()
    
    val_questions = df['question'].iloc[val_idx].tolist()
    val_references = df['reference_answer'].iloc[val_idx].tolist()
    val_students = df['student_answer'].iloc[val_idx].tolist()
    val_scores = df['score'].iloc[val_idx].tolist()
    
    print(f"\nTraining samples: {len(train_idx)}")
    print(f"Validation samples: {len(val_idx)}")
    
    # Create datasets
    train_dataset = SAGDataset(train_questions, train_references, train_students, train_scores, preprocessor)
    val_dataset = SAGDataset(val_questions, val_references, val_students, val_scores, preprocessor)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # Load pretrained embeddings if available
    pretrained_embeddings = None
    if os.path.exists(glove_path):
        pretrained_embeddings = load_glove_embeddings(glove_path, preprocessor.word2idx, embedding_dim=EMBEDDING_DIM)
    else:
        print(f"GloVe not found at {glove_path}, using random embeddings...")
    
    # Create model
    model = SiameseLSTM(
        vocab_size=preprocessor.vocab_size,
        embedding_dim=EMBEDDING_DIM,
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        pretrained_embeddings=pretrained_embeddings
    )
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nTotal parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    
    # Train model
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"\nTraining on {device}")
    
    model = train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LEARNING_RATE, device=device)
    
    # Save preprocessor
    with open('preprocessor.pkl', 'wb') as f:
        pickle.dump(preprocessor, f)
    
    print("\n" + "=" * 60)
    print("TRAINING COMPLETED!")
    print("Model saved to: best_siamese_lstm_model.pth")
    print("Preprocessor saved to: preprocessor.pkl")
    print("=" * 60)
    
    return model, preprocessor


# Run training
if __name__ == "__main__":
    # Update the CSV path if needed
    CSV_PATH = '/kaggle/input/datasets/uppulurimadhuri/semeval2013-task-7-sag/Training_data.csv'  # Your file name
    GLOVE_PATH = '/kaggle/input/datasets/rtatman/glove-global-vectors-for-word-representation/glove.6B.100d.txt'  # Update if you have GloVe
    
    model, preprocessor = main_training(csv_path=CSV_PATH, glove_path=GLOVE_PATH)

LOADING DATASET
Loaded 4969 valid samples
Score distribution:
score
0.0    1138
0.3     499
0.7    1324
1.0    2008
Name: count, dtype: int64

Vocabulary size: 2558
Max sequence length: 150

Training samples: 3975
Validation samples: 994
Loading GloVe embeddings from /kaggle/input/datasets/rtatman/glove-global-vectors-for-word-representation/glove.6B.100d.txt...
GloVe embeddings loaded successfully

Total parameters: 2,697,272
Trainable parameters: 2,697,272

Training on cuda
Epoch 1/20
  Train Loss: 0.1888
  Val Loss: 0.1905
  Pearson: 0.1775, Spearman: 0.1712, MSE: 0.1960
  Saved best model with val loss: 0.1905, Pearson: 0.1775

Epoch 2/20
  Train Loss: 0.1629
  Val Loss: 0.1553
  Pearson: 0.1840, Spearman: 0.1874, MSE: 0.1597
  Saved best model with val loss: 0.1553, Pearson: 0.1840

Epoch 3/20
  Train Loss: 0.1496
  Val Loss: 0.1466
  Pearson: 0.2572, Spearman: 0.2542, MSE: 0.1504
  Saved best model with val loss: 0.1466, Pearson: 0.2572

Epoch 4/20
  Train Loss: 0.1338
  Val Loss

In [18]:
# ============================================
# CELL 2: LOADING MODEL AND EVALUATING TEST INPUT
# ============================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pickle
import re
import string
import os

# ============================================
# CONFIGURATION (Must match training)
# ============================================
MAX_SEQUENCE_LENGTH = 150
EMBEDDING_DIM = 100
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.3

# ============================================
# MODEL ARCHITECTURE (Must match training)
# ============================================

class SiameseLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=EMBEDDING_DIM, hidden_dim=HIDDEN_DIM, 
                 num_layers=NUM_LAYERS, dropout=DROPOUT, pretrained_embeddings=None):
        super(SiameseLSTM, self).__init__()
        
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        
        if pretrained_embeddings is not None:
            self.embedding = nn.Embedding.from_pretrained(pretrained_embeddings, freeze=False)
        else:
            self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.fc = nn.Linear(hidden_dim * 2, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        
    def encode(self, x):
        embedded = self.dropout(self.embedding(x))
        lstm_out, (hidden, cell) = self.lstm(embedded)
        
        hidden_forward = hidden[-2, :, :]
        hidden_backward = hidden[-1, :, :]
        hidden_combined = torch.cat((hidden_forward, hidden_backward), dim=1)
        
        embedding = self.fc(hidden_combined)
        embedding = torch.tanh(embedding)
        return embedding
    
    def forward(self, input_a, input_b):
        embedding_a = self.encode(input_a)
        embedding_b = self.encode(input_b)
        
        cos_sim = nn.functional.cosine_similarity(embedding_a, embedding_b, dim=1)
        cos_sim = (cos_sim + 1) / 2
        
        return cos_sim


# ============================================
# PREPROCESSOR CLASS (Must match training)
# ============================================

class TextPreprocessor:
    def __init__(self, max_sequence_length=MAX_SEQUENCE_LENGTH):
        self.max_sequence_length = max_sequence_length
        self.word2idx = {}
        self.idx2word = {}
        self.vocab_size = 0
        
    def clean_text(self, text):
        if pd.isna(text):
            text = ""
        text = str(text).lower()
        text = re.sub(f'[{re.escape(string.punctuation)}]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    
    def text_to_sequence(self, text):
        words = self.clean_text(text).split()
        sequence = [self.word2idx.get(word, self.word2idx.get('<UNK>', 1)) for word in words]
        if len(sequence) > self.max_sequence_length:
            sequence = sequence[:self.max_sequence_length]
        else:
            sequence = sequence + [self.word2idx.get('<PAD>', 0)] * (self.max_sequence_length - len(sequence))
        return sequence


# ============================================
# INFERENCE FUNCTION
# ============================================

class AnswerGrader:
    def __init__(self, model_path, preprocessor_path, device='cpu'):
        self.device = device
        
        # Load preprocessor
        with open(preprocessor_path, 'rb') as f:
            self.preprocessor = pickle.load(f)
        
        # Load model - FIX: Add weights_only=False for PyTorch 2.6+
        checkpoint = torch.load(model_path, map_location=device, weights_only=False)
        
        # Handle both full checkpoint and state dict only
        if 'model_state_dict' in checkpoint:
            model_state_dict = checkpoint['model_state_dict']
        else:
            model_state_dict = checkpoint
        
        self.model = SiameseLSTM(
            vocab_size=self.preprocessor.vocab_size,
            embedding_dim=EMBEDDING_DIM,
            hidden_dim=HIDDEN_DIM,
            num_layers=NUM_LAYERS,
            dropout=DROPOUT
        )
        self.model.load_state_dict(model_state_dict)
        self.model.to(device)
        self.model.eval()
        
        print(f"Model loaded successfully. Vocab size: {self.preprocessor.vocab_size}")
        
    def grade_answer(self, question, reference_answer, student_answer, max_marks):
        input_a = f"{question} {reference_answer}"
        input_b = student_answer
        
        seq_a = self.preprocessor.text_to_sequence(input_a)
        seq_b = self.preprocessor.text_to_sequence(input_b)
        
        tensor_a = torch.tensor([seq_a], dtype=torch.long).to(self.device)
        tensor_b = torch.tensor([seq_b], dtype=torch.long).to(self.device)
        
        with torch.no_grad():
            similarity = self.model(tensor_a, tensor_b).item()
        
        final_marks = similarity * max_marks
        
        return similarity, final_marks


# ============================================
# PROCESS TEST CSV AND GENERATE OUTPUT
# ============================================

def process_test_csv(csv_path, model_path, preprocessor_path):
    """
    Process test CSV file and generate grading results
    
    Expected CSV columns:
    - Question number (or Question_ID)
    - Passage (or Reference_Answer_Text)
    - Question (or Question_Text)
    - student_answer (or Student_Answer_Text)
    - max_marks
    """
    # Read test data
    test_df = pd.read_csv(csv_path)
    
    # Print column names for debugging
    print("\n" + "="*80)
    print("TEST CSV INFORMATION")
    print("="*80)
    print(f"File loaded: {csv_path}")
    print(f"Number of rows: {len(test_df)}")
    print(f"Available columns: {test_df.columns.tolist()}")
    print("="*80 + "\n")
    
    # Map column names (handle different naming conventions)
    col_mapping = {
        'question_number': ['Question number', 'Question_ID', 'Question_Number', 'Q_ID', 'ID', 'Unnamed: 0'],
        'passage': ['Passage', 'Reference_Answer_Text', 'reference_answer', 'ReferenceAnswer', 'reference'],
        'question': ['Question', 'Question_Text', 'question_text', 'QuestionText', 'q_text'],
        'student_answer': ['student_answer', 'Student_Answer_Text', 'StudentAnswer', 'answer', 'Answer'],
        'max_marks': ['max_marks', 'Max_Marks', 'max_marks', 'marks', 'Marks']
    }
    
    # Find actual column names
    actual_cols = {}
    for target, possibilities in col_mapping.items():
        for col in test_df.columns:
            if col in possibilities or col.lower() in [p.lower() for p in possibilities]:
                actual_cols[target] = col
                break
    
    # If not found, try to infer from column names
    if 'question_number' not in actual_cols:
        # Look for any ID-like column
        for col in test_df.columns:
            if 'id' in col.lower() or 'number' in col.lower():
                actual_cols['question_number'] = col
                break
        if 'question_number' not in actual_cols:
            actual_cols['question_number'] = test_df.columns[0]  # Use first column
    
    if 'passage' not in actual_cols:
        for col in test_df.columns:
            if 'passage' in col.lower() or 'reference' in col.lower():
                actual_cols['passage'] = col
                break
        if 'passage' not in actual_cols:
            actual_cols['passage'] = test_df.columns[1] if len(test_df.columns) > 1 else test_df.columns[0]
    
    if 'question' not in actual_cols:
        for col in test_df.columns:
            if 'question' in col.lower():
                actual_cols['question'] = col
                break
        if 'question' not in actual_cols:
            actual_cols['question'] = test_df.columns[2] if len(test_df.columns) > 2 else test_df.columns[0]
    
    if 'student_answer' not in actual_cols:
        for col in test_df.columns:
            if 'answer' in col.lower() or 'student' in col.lower():
                actual_cols['student_answer'] = col
                break
        if 'student_answer' not in actual_cols:
            actual_cols['student_answer'] = test_df.columns[3] if len(test_df.columns) > 3 else test_df.columns[0]
    
    if 'max_marks' not in actual_cols:
        actual_cols['max_marks'] = 'max_marks'
        if 'max_marks' not in test_df.columns:
            # Add default max_marks column
            test_df['max_marks'] = 10
            actual_cols['max_marks'] = 'max_marks'
    
    print(f"Column mapping used:")
    for key, value in actual_cols.items():
        print(f"  {key} -> '{value}'")
    print()
    
    # Initialize grader
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    grader = AnswerGrader(model_path, preprocessor_path, device)
    
    # Grade each answer
    results = []
    total_marks_obtained = 0
    total_max_marks = 0
    
    print("Grading in progress...")
    print("-"*80)
    
    for idx, row in test_df.iterrows():
        question_num = row[actual_cols['question_number']]
        question = row[actual_cols['question']]
        passage = row[actual_cols['passage']]
        student_answer = row[actual_cols['student_answer']]
        max_marks = float(row[actual_cols['max_marks']])
        
        # Grade the answer
        similarity, marks_obtained = grader.grade_answer(
            question, passage, student_answer, max_marks
        )
        
        results.append({
            'Question Number': question_num,
            'Question': question,
            'Student Answer': student_answer,
            'Marks Obtained': round(marks_obtained, 2),
            'Maximum Marks': max_marks,
            'Similarity Score': round(similarity, 3)
        })
        
        total_marks_obtained += marks_obtained
        total_max_marks += max_marks
    
    # Create results DataFrame
    results_df = pd.DataFrame(results)
    
    # ============================================
    # PRINT OUTPUT TO CONSOLE
    # ============================================
    
    print("\n" + "="*100)
    print("GRADING RESULTS")
    print("="*100)
    print()
    
    # Print header
    print(f"{'Q#':<5} {'Question (truncated)':<60} {'Obtained':<10} {'Max':<8} {'Similarity':<10}")
    print("-"*100)
    
    # Print each result
    for _, row in results_df.iterrows():
        q_num = row['Question Number']
        question_text = str(row['Question'])[:57] + "..." if len(str(row['Question'])) > 57 else str(row['Question'])
        marks_obt = row['Marks Obtained']
        marks_max = row['Maximum Marks']
        similarity = row['Similarity Score']
        
        print(f"{q_num:<5} {question_text:<60} {marks_obt:<10.2f} {marks_max:<8.0f} {similarity:<10.3f}")
    
    print()
    print("="*100)
    print(f"TOTAL SCORE: {total_marks_obtained:.2f} out of {total_max_marks:.0f}")
    print(f"PERCENTAGE: {(total_marks_obtained/total_max_marks*100):.1f}%")
    print("="*100)
    
    # Print detailed table with full answers
    print("\n" + "="*100)
    print("DETAILED RESULTS (Full Answers)")
    print("="*100)
    print()
    
    for i, row in results_df.iterrows():
        print(f"\n{'='*80}")
        print(f"QUESTION #{row['Question Number']}")
        print(f"{'='*80}")
        print(f"Question: {row['Question']}")
        print(f"Student Answer: {row['Student Answer']}")
        print(f"\nSimilarity Score: {row['Similarity Score']:.3f}")
        print(f"Marks: {row['Marks Obtained']:.2f} / {row['Maximum Marks']:.0f}")
        print()
    
    print("="*100)
    print(f"FINAL SUMMARY: {total_marks_obtained:.2f} / {total_max_marks:.0f} ({total_marks_obtained/total_max_marks*100:.1f}%)")
    print("="*100)
    
    # Also save to CSV for record
    results_df.to_csv('grading_results.csv', index=False)
    print("\nResults also saved to 'grading_results.csv'")
    
    return results_df, total_marks_obtained, total_max_marks


# ============================================
# MAIN EXECUTION
# ============================================

if __name__ == "__main__":
    MODEL_PATH = 'best_siamese_lstm_model.pth'
    PREPROCESSOR_PATH = 'preprocessor.pkl'
    
    # UPDATE THIS PATH TO YOUR TEST CSV FILE
    TEST_CSV_PATH = '/kaggle/input/datasets/lakshmir03/test-eval/Test_Ans_Eval - Sheet1.csv'  # <-- CHANGE THIS TO YOUR FILE PATH
    
    # Check if model files exist
    if not os.path.exists(MODEL_PATH):
        print(f"ERROR: Model file '{MODEL_PATH}' not found!")
        print("Please run Cell 1 first to train and save the model.")
        exit(1)
    
    if not os.path.exists(PREPROCESSOR_PATH):
        print(f"ERROR: Preprocessor file '{PREPROCESSOR_PATH}' not found!")
        print("Please run Cell 1 first to train and save the model.")
        exit(1)
    
    if not os.path.exists(TEST_CSV_PATH):
        print(f"ERROR: Test CSV file '{TEST_CSV_PATH}' not found!")
        print("Please provide the correct path to your test CSV file.")
        print("\nExpected CSV columns:")
        print("  - Question number (or Question_ID, ID, etc.)")
        print("  - Passage (or Reference_Answer_Text)")
        print("  - Question (or Question_Text)")
        print("  - student_answer (or Student_Answer_Text)")
        print("  - max_marks")
        exit(1)
    
    # Process the test CSV
    try:
        results_df, total_obtained, total_max = process_test_csv(
            TEST_CSV_PATH, 
            MODEL_PATH, 
            PREPROCESSOR_PATH
        )
        
    except Exception as e:
        print(f"\nError during grading: {str(e)}")
        import traceback
        traceback.print_exc()


TEST CSV INFORMATION
File loaded: /kaggle/input/datasets/lakshmir03/test-eval/Test_Ans_Eval - Sheet1.csv
Number of rows: 15
Available columns: ['Question number', 'Question', 'Passage', 'student_answer', 'max_marks']

Column mapping used:
  question_number -> 'Question number'
  passage -> 'Passage'
  question -> 'Question'
  student_answer -> 'student_answer'
  max_marks -> 'max_marks'

Model loaded successfully. Vocab size: 2558
Grading in progress...
--------------------------------------------------------------------------------

GRADING RESULTS

Q#    Question (truncated)                                         Obtained   Max      Similarity
----------------------------------------------------------------------------------------------------
1     Who were Tara and Sitara?                                    1.08       2        0.541     
2     What did Sitara enjoy doing?                                 1.90       2        0.951     
3     Where did they live?                     

# SBERT

In [1]:
pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.4/512.4 kB 9.6 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.2.3
    Uninstalling sentence-transformers-5.2.3:
      Successfully uninstalled sentence-transformers-5.2.3
Note: you may need to restart the kernel to use updated packages.


In [2]:
# ============================================
# CELL 1: SETUP AND MODEL SAVING
# Sentence-BERT based ASAG System
# ============================================

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
import re
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize
import nltk
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)

# ============================================
# CONFIGURATION
# ============================================
class Config:
    MODEL_NAME = 'all-MiniLM-L6-v2'  # Fast and effective SBERT model
    SAVE_DIR = './asag_model'  # Directory to save model and components
    TOP_K_SENTENCES = 3  # Number of relevant sentences to extract
    SEMANTIC_WEIGHT = 0.8  # Weight for semantic similarity
    KEYWORD_WEIGHT = 0.2   # Weight for keyword overlap
    LENGTH_PENALTY_STRENGTH = 0.3  # How strongly to penalize long answers
    MIN_KEYWORD_OVERLAP = 0.1  # Minimum keyword overlap threshold
    STOPWORDS = set(stopwords.words('english'))

# Create save directory if it doesn't exist
os.makedirs(Config.SAVE_DIR, exist_ok=True)

# ============================================
# TEXT PREPROCESSING
# ============================================

class TextPreprocessor:
    @staticmethod
    def clean_text(text):
        """Basic text cleaning"""
        if pd.isna(text):
            return ""
        text = str(text).lower()
        text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    
    @staticmethod
    def extract_keywords(text):
        """Extract meaningful keywords (remove stopwords)"""
        words = TextPreprocessor.clean_text(text).split()
        keywords = [w for w in words if w not in Config.STOPWORDS and len(w) > 2]
        return set(keywords)
    
    @staticmethod
    def calculate_keyword_overlap(text1, text2):
        """Calculate Jaccard similarity between keyword sets"""
        keywords1 = TextPreprocessor.extract_keywords(text1)
        keywords2 = TextPreprocessor.extract_keywords(text2)
        
        if not keywords1 or not keywords2:
            return 0.0
        
        intersection = len(keywords1 & keywords2)
        union = len(keywords1 | keywords2)
        
        return intersection / union if union > 0 else 0.0


# ============================================
# SENTENCE EXTRACTION MODULE
# ============================================

class SentenceExtractor:
    def __init__(self, model):
        self.model = model
        
    def extract_relevant_sentences(self, passage, question, top_k=Config.TOP_K_SENTENCES):
        """
        Extract top-k sentences from passage most relevant to the question
        """
        if not passage or pd.isna(passage):
            return [], []
        
        # Split passage into sentences
        sentences = sent_tokenize(passage)
        
        if not sentences:
            return [], []
        
        # Compute embeddings for question and all sentences
        question_embedding = self.model.encode(question, convert_to_tensor=True)
        sentence_embeddings = self.model.encode(sentences, convert_to_tensor=True)
        
        # Calculate cosine similarities
        similarities = util.cos_sim(question_embedding, sentence_embeddings)[0]
        
        # Get top-k relevant sentences
        top_indices = torch.topk(similarities, min(top_k, len(sentences))).indices.tolist()
        relevant_sentences = [sentences[i] for i in top_indices]
        relevance_scores = [similarities[i].item() for i in top_indices]
        
        return relevant_sentences, relevance_scores


# ============================================
# SCORING MODULE
# ============================================

class AnswerScorer:
    def __init__(self, model):
        self.model = model
        
    def compute_semantic_similarity(self, text1, text2):
        """Compute cosine similarity between two texts using SBERT"""
        if not text1 or not text2:
            return 0.0
        
        emb1 = self.model.encode(text1, convert_to_tensor=True)
        emb2 = self.model.encode(text2, convert_to_tensor=True)
        
        similarity = util.cos_sim(emb1, emb2).item()
        return max(0.0, min(1.0, similarity))
    
    def apply_length_penalty(self, student_answer, reference_text, similarity_score):
        """
        Apply penalty if student answer is significantly longer than reference
        """
        student_len = len(student_answer.split())
        ref_len = len(reference_text.split())
        
        if ref_len == 0:
            return similarity_score
        
        length_ratio = student_len / ref_len
        
        # Apply penalty only if answer is much longer (ratio > 1.5)
        if length_ratio > 1.5:
            penalty = min(Config.LENGTH_PENALTY_STRENGTH, (length_ratio - 1.5) * 0.1)
            return similarity_score * (1 - penalty)
        
        return similarity_score
    
    def check_relevance(self, student_answer, question):
        """
        Quick relevance check - if student answer is completely irrelevant to question
        """
        # Encode both
        q_emb = self.model.encode(question, convert_to_tensor=True)
        s_emb = self.model.encode(student_answer, convert_to_tensor=True)
        
        relevance = util.cos_sim(q_emb, s_emb).item()
        
        # If relevance is extremely low (< 0.1), consider it irrelevant
        return relevance > 0.1, relevance
    
    def compute_final_score(self, student_answer, question, relevant_sentences, max_marks):
        """
        Compute final score using all components
        """
        # Quick relevance check
        is_relevant, relevance_score = self.check_relevance(student_answer, question)
        
        if not is_relevant:
            return 0.0, {
                'relevance': relevance_score,
                'semantic_similarity': 0.0,
                'keyword_overlap': 0.0,
                'final_score': 0.0
            }
        
        # Combine relevant sentences into a single reference
        reference_text = " ".join(relevant_sentences)
        
        # Compute semantic similarity
        semantic_sim = self.compute_semantic_similarity(student_answer, reference_text)
        
        # Compute keyword overlap
        keyword_overlap = TextPreprocessor.calculate_keyword_overlap(
            student_answer, reference_text
        )
        
        # Combine scores
        combined_score = (Config.SEMANTIC_WEIGHT * semantic_sim + 
                         Config.KEYWORD_WEIGHT * keyword_overlap)
        
        # Apply length penalty
        final_score = self.apply_length_penalty(student_answer, reference_text, combined_score)
        
        # Scale to max_marks
        scaled_score = final_score * max_marks
        
        # Clamp to [0, max_marks]
        scaled_score = max(0.0, min(max_marks, scaled_score))
        
        return scaled_score, {
            'relevance': relevance_score,
            'semantic_similarity': semantic_sim,
            'keyword_overlap': keyword_overlap,
            'final_score': final_score,
            'reference_text': reference_text[:200] + "..." if len(reference_text) > 200 else reference_text
        }


# ============================================
# MAIN ASAG SYSTEM
# ============================================

class AutomaticShortAnswerGrader:
    def __init__(self, model_name=Config.MODEL_NAME, load_saved=True):
        self.model_name = model_name
        self.model_path = os.path.join(Config.SAVE_DIR, 'sbert_model.pkl')
        self.config_path = os.path.join(Config.SAVE_DIR, 'config.pkl')
        
        if load_saved and os.path.exists(self.model_path):
            print(f"Loading saved model from {self.model_path}...")
            self._load_model()
        else:
            print(f"Loading Sentence-BERT model: {model_name}...")
            self.model = SentenceTransformer(model_name)
            self._save_model()
            
        self.extractor = SentenceExtractor(self.model)
        self.scorer = AnswerScorer(self.model)
        print("Model ready!")
        
    def _save_model(self):
        """Save the model and configuration"""
        print(f"Saving model to {self.model_path}...")
        # Save the model
        self.model.save(self.model_path.replace('.pkl', ''))
        # Save configuration
        with open(self.config_path, 'wb') as f:
            pickle.dump({
                'model_name': self.model_name,
                'config': {
                    'TOP_K_SENTENCES': Config.TOP_K_SENTENCES,
                    'SEMANTIC_WEIGHT': Config.SEMANTIC_WEIGHT,
                    'KEYWORD_WEIGHT': Config.KEYWORD_WEIGHT,
                    'LENGTH_PENALTY_STRENGTH': Config.LENGTH_PENALTY_STRENGTH
                }
            }, f)
        print("Model saved successfully!")
        
    def _load_model(self):
        """Load saved model and configuration"""
        # Load the model
        self.model = SentenceTransformer(self.model_path.replace('.pkl', ''))
        # Load configuration
        with open(self.config_path, 'rb') as f:
            saved_data = pickle.load(f)
        print(f"Loaded model: {saved_data['model_name']}")
        print(f"Loaded config: {saved_data['config']}")
        
    def grade_single_answer(self, question, passage, student_answer, max_marks):
        """
        Grade a single answer and return score with explanations
        """
        # Step 1: Extract relevant sentences from passage
        relevant_sentences, relevance_scores = self.extractor.extract_relevant_sentences(
            passage, question
        )
        
        if not relevant_sentences:
            # No relevant sentences found, answer is likely irrelevant
            return 0.0, {
                'relevant_sentences': [],
                'relevance_scores': [],
                'semantic_similarity': 0.0,
                'keyword_overlap': 0.0,
                'final_score': 0.0,
                'explanation': "No relevant sentences found in passage."
            }
        
        # Step 2: Compute final score
        final_score, scores_dict = self.scorer.compute_final_score(
            student_answer, question, relevant_sentences, max_marks
        )
        
        # Add explanation
        scores_dict['relevant_sentences'] = relevant_sentences
        scores_dict['relevance_scores'] = relevance_scores
        scores_dict['explanation'] = self._generate_explanation(scores_dict)
        
        return final_score, scores_dict
    
    def _generate_explanation(self, scores_dict):
        """Generate human-readable explanation of the scoring"""
        explanation = f"""
        Scoring Breakdown:
        - Semantic Similarity: {scores_dict['semantic_similarity']:.3f} (weight: {Config.SEMANTIC_WEIGHT})
        - Keyword Overlap: {scores_dict['keyword_overlap']:.3f} (weight: {Config.KEYWORD_WEIGHT})
        - Combined Score: {scores_dict['final_score']:.3f}
        - Relevance to Question: {scores_dict.get('relevance', 0):.3f}
        
        Reference sentences used:
        {chr(10).join(f'  • {s}' for s in scores_dict.get('relevant_sentences', [])[:3])}
        """
        return explanation
    
    def grade_batch(self, test_df):
        """
        Grade a batch of answers from a DataFrame
        Expected columns: Question number, Passage, Question, student_answer, max_marks
        """
        results = []
        
        for idx, row in test_df.iterrows():
            question_num = row['Question number'] if 'Question number' in row else idx + 1
            passage = row['Passage']
            question = row['Question']
            student_answer = row['student_answer']
            max_marks = float(row['max_marks'])
            
            score, details = self.grade_single_answer(
                question, passage, student_answer, max_marks
            )
            
            results.append({
                'Question Number': question_num,
                'Question': question,
                'Student Answer': student_answer,
                'Marks Obtained': round(score, 2),
                'Maximum Marks': max_marks,
                'Semantic Similarity': round(details['semantic_similarity'], 3),
                'Keyword Overlap': round(details['keyword_overlap'], 3),
                'Relevant Sentences': ' | '.join(details.get('relevant_sentences', [])[:2])
            })
        
        return pd.DataFrame(results)


# ============================================
# INITIALIZATION AND SAVING
# ============================================

# Initialize and save the grader
print("="*60)
print("INITIALIZING ASAG SYSTEM")
print("="*60)

grader = AutomaticShortAnswerGrader(load_saved=False)  # Set to True to load existing

print("\n" + "="*60)
print("ASAG System Ready!")
print("="*60)
print(f"Model: {Config.MODEL_NAME}")
print(f"Model saved at: {Config.SAVE_DIR}/sbert_model.pkl")
print(f"Top-K Sentences: {Config.TOP_K_SENTENCES}")
print(f"Semantic Weight: {Config.SEMANTIC_WEIGHT}")
print(f"Keyword Weight: {Config.KEYWORD_WEIGHT}")
print("="*60)




INITIALIZING ASAG SYSTEM
Loading Sentence-BERT model: all-MiniLM-L6-v2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Saving model to ./asag_model/sbert_model.pkl...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!
Model ready!

ASAG System Ready!
Model: all-MiniLM-L6-v2
Model saved at: ./asag_model/sbert_model.pkl
Top-K Sentences: 3
Semantic Weight: 0.8
Keyword Weight: 0.2


In [6]:
# ============================================
# CELL 2: LOAD MODEL AND PROCESS CSV (WITH ANSWER COLUMN)
# ============================================

import pandas as pd
import os
import numpy as np
from sentence_transformers import SentenceTransformer, util
import re
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords
import nltk
import warnings
warnings.filterwarnings('ignore')

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)

# ============================================
# CONFIGURATION
# ============================================
class Config:
    TOP_K_SENTENCES = 3
    SEMANTIC_WEIGHT = 0.9  # Increased weight for semantic similarity
    KEYWORD_WEIGHT = 0.1   # Decreased weight for keyword overlap
    STOPWORDS = set(stopwords.words('english'))


# ============================================
# TEXT PREPROCESSING
# ============================================
class TextPreprocessor:
    @staticmethod
    def clean_text(text):
        if pd.isna(text):
            return ""
        text = str(text).lower()
        text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    
    @staticmethod
    def extract_keywords(text):
        words = TextPreprocessor.clean_text(text).split()
        keywords = [w for w in words if w not in Config.STOPWORDS and len(w) > 2]
        return set(keywords)
    
    @staticmethod
    def calculate_keyword_overlap(text1, text2):
        keywords1 = TextPreprocessor.extract_keywords(text1)
        keywords2 = TextPreprocessor.extract_keywords(text2)
        
        if not keywords1 or not keywords2:
            return 0.0
        
        intersection = len(keywords1 & keywords2)
        union = len(keywords1 | keywords2)
        
        return intersection / union if union > 0 else 0.0


# ============================================
# SENTENCE EXTRACTION
# ============================================
def extract_relevant_sentences(passage, question, model, top_k=Config.TOP_K_SENTENCES):
    """
    Extract relevant sentences using cosine similarity
    """
    if not passage or pd.isna(passage):
        return []
    
    # Split passage into sentences
    sentences = sent_tokenize(str(passage))
    
    if not sentences:
        return []
    
    # Adjust top_k
    actual_k = min(top_k, len(sentences))
    
    try:
        # Encode question and sentences
        question_emb = model.encode(question, convert_to_numpy=True)
        sentence_embs = model.encode(sentences, convert_to_numpy=True)
        
        # Calculate similarities
        similarities = []
        for sent_emb in sentence_embs:
            sim = util.cos_sim(question_emb, sent_emb)
            similarities.append(sim)
        
        # Get top-k indices
        similarities = np.array(similarities).flatten()
        top_indices = similarities.argsort()[-actual_k:][::-1]
        
        relevant_sentences = [sentences[i] for i in top_indices]
        return relevant_sentences
        
    except Exception as e:
        return []


# ============================================
# SCORING MODULE
# ============================================
class AnswerScorer:
    def __init__(self, model):
        self.model = model
        
    def compute_semantic_similarity(self, text1, text2):
        if not text1 or not text2 or len(text1.strip()) == 0 or len(text2.strip()) == 0:
            return 0.0
        
        try:
            emb1 = self.model.encode(text1, convert_to_numpy=True)
            emb2 = self.model.encode(text2, convert_to_numpy=True)
            
            similarity = util.cos_sim(emb1, emb2)
            return float(max(0.0, min(1.0, similarity)))
        except:
            return 0.0
    
    def compute_final_score(self, student_answer, relevant_sentences, max_marks):
        # Handle empty answers
        if not student_answer or len(student_answer.strip()) < 3:
            return 0.0
        
        # Handle no relevant sentences
        if not relevant_sentences:
            return 0.0
        
        reference_text = " ".join(relevant_sentences)
        
        # Compute semantic similarity
        semantic_sim = self.compute_semantic_similarity(student_answer, reference_text)
        
        # Compute keyword overlap (reduced importance)
        keyword_overlap = TextPreprocessor.calculate_keyword_overlap(student_answer, reference_text)
        
        # Combine scores with higher weight on semantic similarity
        final_score = (Config.SEMANTIC_WEIGHT * semantic_sim + 
                      Config.KEYWORD_WEIGHT * keyword_overlap)
        
        # Scale to max_marks
        scaled_score = final_score * max_marks
        scaled_score = max(0.0, min(max_marks, scaled_score))
        
        return scaled_score, semantic_sim, keyword_overlap


# ============================================
# MAIN GRADER CLASS
# ============================================
class AutomaticShortAnswerGrader:
    def __init__(self):
        print("Loading Sentence-BERT model...")
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.scorer = AnswerScorer(self.model)
        print("Model ready!\n")
        
    def grade_answer(self, question, passage, student_answer, max_marks):
        if pd.isna(question) or pd.isna(passage) or pd.isna(student_answer):
            return 0.0, 0.0, 0.0
        
        relevant_sentences = extract_relevant_sentences(passage, question, self.model)
        score, semantic, keyword = self.scorer.compute_final_score(
            student_answer, relevant_sentences, max_marks
        )
        
        return score, semantic, keyword


# ============================================
# INITIALIZE GRADER
# ============================================
grader = AutomaticShortAnswerGrader()

# ============================================
# PROCESS TEST CSV WITH CORRECT OUTPUT FORMAT
# ============================================

def process_test_csv(csv_path):
    """
    Process test CSV and generate grades in the required format
    """
    # Read test data
    print(f"Loading test data from: {csv_path}")
    test_df = pd.read_csv(csv_path)
    
    print(f"Found {len(test_df)} questions to grade\n")
    
    # Ensure max_marks is numeric
    if 'max_marks' in test_df.columns:
        test_df['max_marks'] = pd.to_numeric(test_df['max_marks'], errors='coerce').fillna(10)
    else:
        test_df['max_marks'] = 10
    
    # Grade all answers
    results = []
    total_marks_obtained = 0
    total_max_marks = 0
    
    print("="*120)
    print("GRADING IN PROGRESS")
    print("="*120)
    
    for idx, row in test_df.iterrows():
        question_num = row['Question number']
        passage = row['Passage']
        question = row['Question']
        student_answer = row['student_answer']
        max_marks = float(row['max_marks'])
        
        # Grade the answer
        try:
            score, semantic, keyword = grader.grade_answer(
                question, passage, student_answer, max_marks
            )
            print(f"Q{question_num}: {score:.2f}/{max_marks:.0f} (Semantic: {semantic:.3f})")
        except Exception as e:
            print(f"Q{question_num}: Error - {str(e)[:50]}")
            score, semantic, keyword = 0.0, 0.0, 0.0
        
        results.append({
            'Question number': question_num,
            'Question': question,
            'Student Answer': student_answer,  # Added student answer column
            'allotted_mark': round(score, 2),
            'maximum_mark': max_marks
        })
        
        total_marks_obtained += score
        total_max_marks += max_marks
    
    results_df = pd.DataFrame(results)
    
    # Print results in the exact format requested (with answer column)
    print("\n" + "="*140)
    print("GRADING RESULTS")
    print("="*140)
    
    # Display as table with answer column
    print(f"\n{'Q#':<6} {'Question':<45} {'Student Answer':<40} {'Marks':<12} {'Max':<8}")
    print("-"*120)
    
    for _, row in results_df.iterrows():
        # Truncate long text
        question_text = str(row['Question'])
        if len(question_text) > 42:
            question_text = question_text[:39] + "..."
        
        answer_text = str(row['Student Answer'])
        if len(answer_text) > 37:
            answer_text = answer_text[:34] + "..."
        
        print(f"{row['Question number']:<6} {question_text:<45} {answer_text:<40} {row['allotted_mark']:<12.2f} {row['maximum_mark']:<8.0f}")
    
    print("\n" + "="*140)
    print(f"TOTAL MARKS SCORED: {total_marks_obtained:.2f} out of {total_max_marks:.0f}")
    print(f"PERCENTAGE: {(total_marks_obtained/total_max_marks*100):.1f}%")
    print("="*140)
    
    # Save to CSV with all required columns
    output_df = results_df[['Question number', 'Question', 'Student Answer', 'allotted_mark', 'maximum_mark']]
    output_path = 'grading_results.csv'
    output_df.to_csv(output_path, index=False)
    print(f"\nResults saved to: {output_path}")
    print(f"Columns: Question number, Question, Student Answer, allotted_mark, maximum_mark")
    
    # Print detailed summary with answers
    print("\n" + "="*80)
    print("DETAILED SUMMARY (WITH ANSWERS)")
    print("="*80)
    for _, row in results_df.iterrows():
        print(f"\nQ{row['Question number']}: {row['Question']}")
        print(f"  Answer: {row['Student Answer']}")
        print(f"  Score: {row['allotted_mark']:.2f} / {row['maximum_mark']:.0f}")
    
    print("\n" + "="*80)
    print(f"TOTAL: {total_marks_obtained:.2f} / {total_max_marks:.0f}")
    print("="*80)
    
    return results_df


# ============================================
# MAIN EXECUTION
# ============================================

if __name__ == "__main__":
    # Path to your test CSV file
    TEST_CSV_PATH = '/kaggle/input/datasets/lakshmir03/test-eval/Test_Ans_Eval - Sheet1.csv'
    
    if os.path.exists(TEST_CSV_PATH):
        results_df = process_test_csv(TEST_CSV_PATH)
    else:
        print(f"Test CSV file '{TEST_CSV_PATH}' not found!")
        print("Please update the path to your CSV file.")

Loading Sentence-BERT model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model ready!

Loading test data from: /kaggle/input/datasets/lakshmir03/test-eval/Test_Ans_Eval - Sheet1.csv
Found 15 questions to grade

GRADING IN PROGRESS
Q1: 1.05/2 (Semantic: 0.568)
Q2: 0.97/2 (Semantic: 0.535)
Q3: 1.27/2 (Semantic: 0.684)
Q4: 1.42/3 (Semantic: 0.519)
Q5: 2.25/3 (Semantic: 0.774)
Q6: 0.71/2 (Semantic: 0.383)
Q7: 1.77/3 (Semantic: 0.614)
Q8: 2.47/4 (Semantic: 0.656)
Q9: 1.51/4 (Semantic: 0.419)
Q10: 2.09/4 (Semantic: 0.557)
Q11: 2.08/5 (Semantic: 0.456)
Q12: 2.62/5 (Semantic: 0.568)
Q13: 1.34/4 (Semantic: 0.366)
Q14: 0.94/2 (Semantic: 0.500)
Q15: 1.07/3 (Semantic: 0.389)

GRADING RESULTS

Q#     Question                                      Student Answer                           Marks        Max     
------------------------------------------------------------------------------------------------------------------------
1      Who were Tara and Sitara?                     They were two ducks.                     1.05         2       
2      What did Sitara enjoy d

# LLM

In [ ]:
api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not set in environment variables")

client = Groq(api_key=api_key)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 8.1 MB/s eta 0:00:00
Groq setup complete


In [4]:
# ============================
# CELL 2: GROQ ASAG GRADER
# ============================

import pandas as pd
import json

# ============================
# CORE FUNCTION
# ============================

def evaluate_answer(question, passage, student_answer, max_marks):
    prompt = f"""
You are an expert examiner grading short answers.

Your grading must be FAIR, CONSISTENT, and NOT overly strict.

----------------------------------------
QUESTION:
{question}

PASSAGE:
{passage}

STUDENT ANSWER:
{student_answer}

MAX MARKS:
{max_marks}
----------------------------------------

GRADING RUBRIC:

1. FULL MARKS:
- If the answer correctly captures the key idea from the passage
- Even if the answer is short or paraphrased

2. PARTIAL MARKS:
- If the answer is incomplete but contains some correct information

3. LOW MARKS:
- If the answer is vaguely related but mostly incorrect

4. ZERO:
- If the answer is completely incorrect or irrelevant

IMPORTANT RULES:

- Do NOT penalize for short answers if they are correct
- Do NOT expect long explanations
- Focus on correctness, not wording
- If the core fact is correct → give high marks
- If multiple key points exist → distribute marks proportionally

----------------------------------------

Return ONLY JSON:

{{
  "marks": <float>,
  "reason": "<short explanation>"
}}

Ensure:
- marks is between 0 and {max_marks}
- no extra text
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are a strict and fair examiner."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    output = response.choices[0].message.content.strip()

    try:
        result = json.loads(output)
        marks = float(result["marks"])
        reason = result["reason"]
    except:
        marks = 0
        reason = "Parsing error"

    # clamp
    marks = max(0, min(marks, max_marks))

    return marks, reason

# ============================
# LOAD CSV
# ============================

df = pd.read_csv("/kaggle/input/datasets/lakshmir03/test-eval/Test_Ans_Eval - Sheet1.csv")

# ============================
# EVALUATION
# ============================

results = []
total_scored = 0
total_max = 0

for _, row in df.iterrows():

    marks, reason = evaluate_answer(
        row["Question"],
        row["Passage"],
        row["student_answer"],
        row["max_marks"]
    )

    total_scored += marks
    total_max += row["max_marks"]

    results.append({
        "Question Number": row["Question number"],
        "Question": row["Question"],
        "Answer": row["student_answer"],
        "Marks Awarded": round(marks, 2),
        "Max Marks": row["max_marks"],
        "Reason": reason
    })

# ============================
# DISPLAY
# ============================

result_df = pd.DataFrame(results)
display(result_df)

print("\nTotal Score:", round(total_scored, 2), "out of", total_max)

,Question Number,Question,Answer,Marks Awarded,Max Marks,Reason
0,1,Who were Tara and Sitara?,They were two ducks.,2.0,2,Correctly captures the key idea from the passa...
1,2,What did Sitara enjoy doing?,Sitara liked to fly.,0.0,2,The answer is completely incorrect and does no...
2,3,Where did they live?,They lived in a pond.,2.0,2,Correctly captures the key idea from the passage
3,4,Describe their friendship.,They were friends but did different things.,2.0,3,The answer partially captures the key idea fro...
4,5,What is photosynthesis?,"Plants make food using sunlight, water, and ca...",2.5,3,The answer correctly captures the key idea fro...
5,6,When does photosynthesis occur?,It happens at night mostly.,0.0,2,The answer is completely incorrect and contrad...
6,7,Why is chlorophyll important?,It helps plants absorb sunlight.,2.0,3,The answer correctly captures the key idea fro...
7,8,What are the products of photosynthesis?,Oxygen and carbon dioxide are produced.,2.0,4,"The answer is partially correct, as it mention..."
8,9,What problem did minority businesses face?,They didn’t get big company contracts.,3.5,4,The answer correctly captures the key idea fro...
9,10,What action did Congress take?,Companies had to include minority businesses.,3.5,4,The answer correctly captures the key idea fro...



Total Score: 29.0 out of 48
